# The Checkmate Framework
## A Systematic On-Chain Bitcoin Trading Model

*Based on the Masterclass Series by James Check*

---

This notebook implements and tests the Checkmate Framework - a rules-based trading model that synthesizes multiple on-chain metrics into a composite score to identify accumulation and distribution zones.

**Core Philosophy:** Human behavior in markets is a constant we can rely on, and FOMO is one hell of a drug.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Data directory
DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"
print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")

## 1. Data Loading

Load all metrics required for the Checkmate Framework.

In [ ]:
def load_metric(name: str) -> pd.DataFrame:
    """Load a metric from parquet file."""
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        print(f"  ✗ {name} not found")
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if hasattr(df['time'].dt, 'tz') and df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    print(f"  ✓ {name}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")
    return df

# Framework metrics
METRICS_NEEDED = {
    # Layer 1: Investor Profitability
    'mvrv': 'mvrv',
    'mvrv_sth': 'mvrv_sth',  # STH-MVRV
    'mvrv_lth': 'mvrv_lth',  # LTH-MVRV
    'nupl': 'nupl',
    'aviv': 'aviv',
    
    # Layer 2: Spending Behavior  
    'sopr': 'sopr',
    'sopr_adjusted': 'sopr_adjusted',  # aSOPR
    'realized_profit': 'realized_profit',
    'realized_loss': 'realized_loss',
    'cdd': 'cdd',
    'sell_side_risk': 'sell_side_risk',
    
    # Layer 3: Market Structure
    'supply_lth': 'supply_lth',
    'supply_sth': 'supply_sth',
    'exchange_balance': 'exchange_balance',
    'puell_multiple': 'puell_multiple',
    
    # Price data
    'price': 'price',
    'realized_price': 'realized_price',
    'realized_price_sth': 'realized_price_sth',
    'realized_price_lth': 'realized_price_lth',
}

print("Loading metrics...")
data = {}
for key, metric in METRICS_NEEDED.items():
    data[key] = load_metric(metric)

## 2. Framework Configuration

Define the scoring thresholds and weights from the Checkmate Framework.

In [ ]:
# ==============================================================================
# CHECKMATE FRAMEWORK CONFIGURATION
# ==============================================================================

# Layer 1: Investor Profitability Indicators (80% weight)
LAYER1_CONFIG = {
    'mvrv': {
        'weight': 0.25,
        'bullish': 1.0,      # < 1.0 = bullish
        'bearish': 2.4,      # > 2.4 = bearish
        'extreme_bullish': 0.8,
        'extreme_bearish': 3.0,
    },
    'mvrv_sth': {
        'weight': 0.15,
        'bullish': 1.0,
        'bearish': 1.4,
        'extreme_bullish': 0.75,
        'extreme_bearish': 1.6,
    },
    'mvrv_lth': {
        'weight': 0.15,
        'bullish': 1.5,
        'bearish': 3.5,
        'extreme_bullish': 1.0,
        'extreme_bearish': 4.0,
    },
    'nupl': {
        'weight': 0.15,
        'bullish': 0.25,
        'bearish': 0.6,
        'extreme_bullish': 0.0,
        'extreme_bearish': 0.75,
    },
    'aviv': {
        'weight': 0.10,
        'bullish': 1.0,
        'bearish': 1.5,
        'extreme_bullish': 0.75,
        'extreme_bearish': 2.0,
    },
}

# Layer 2: Spending Behavior Indicators
LAYER2_CONFIG = {
    'sopr': {
        'weight': 0.15,
        'bullish': 1.0,
        'bearish': 1.05,
        'extreme_bullish': 0.95,
        'extreme_bearish': 1.10,
    },
    'sopr_adjusted': {
        'weight': 0.10,
        'bullish': 1.0,
        'bearish': 1.08,
        'extreme_bullish': 0.95,
        'extreme_bearish': 1.15,
    },
    'sell_side_risk': {
        'weight': 0.10,
        'bullish': 0.001,   # < 0.1%
        'bearish': 0.005,   # > 0.5%
        'extreme_bullish': 0.0005,
        'extreme_bearish': 0.01,
    },
}

# Layer 3: Market Structure Indicators  
LAYER3_CONFIG = {
    'puell_multiple': {
        'weight': 0.05,
        'bullish': 0.5,
        'bearish': 4.0,
        'extreme_bullish': 0.3,
        'extreme_bearish': 6.0,
    },
}

# Combine all configs
ALL_CONFIG = {**LAYER1_CONFIG, **LAYER2_CONFIG, **LAYER3_CONFIG}

print("Framework Configuration:")
print(f"  Layer 1 (Profitability): {len(LAYER1_CONFIG)} metrics, {sum(c['weight'] for c in LAYER1_CONFIG.values()):.0%} weight")
print(f"  Layer 2 (Spending): {len(LAYER2_CONFIG)} metrics, {sum(c['weight'] for c in LAYER2_CONFIG.values()):.0%} weight")
print(f"  Layer 3 (Structure): {len(LAYER3_CONFIG)} metrics, {sum(c['weight'] for c in LAYER3_CONFIG.values()):.0%} weight")
print(f"  Total: {sum(c['weight'] for c in ALL_CONFIG.values()):.0%}")

## 3. Scoring Functions

Convert raw metric values to standardized scores (-2 to +2).

In [ ]:
def score_metric(value: float, config: dict) -> float:
    """
    Convert a metric value to a score from -2 (extreme bullish) to +2 (extreme bearish).
    
    Score interpretation:
      +2: Extreme Bearish (Euphoria/Distribution)  
      +1: Moderately Bearish (Overheated)
       0: Neutral (Fair Value)
      -1: Moderately Bullish (Undervalued)
      -2: Extreme Bullish (Capitulation/Accumulation)
    """
    if pd.isna(value):
        return 0  # Neutral if no data
    
    bullish = config['bullish']
    bearish = config['bearish']
    extreme_bullish = config.get('extreme_bullish', bullish * 0.75)
    extreme_bearish = config.get('extreme_bearish', bearish * 1.25)
    
    # Calculate midpoint (neutral zone)
    midpoint = (bullish + bearish) / 2
    
    if value <= extreme_bullish:
        return -2.0
    elif value <= bullish:
        # Linear interpolation from -2 to -1
        return -2.0 + (value - extreme_bullish) / (bullish - extreme_bullish)
    elif value <= midpoint:
        # Linear interpolation from -1 to 0
        return -1.0 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        # Linear interpolation from 0 to +1
        return 0.0 + (value - midpoint) / (bearish - midpoint)
    elif value <= extreme_bearish:
        # Linear interpolation from +1 to +2
        return 1.0 + (value - bearish) / (extreme_bearish - bearish)
    else:
        return 2.0

# Test scoring
print("Test MVRV scoring:")
for val in [0.7, 0.9, 1.0, 1.5, 2.0, 2.4, 3.0, 3.5]:
    score = score_metric(val, LAYER1_CONFIG['mvrv'])
    print(f"  MVRV {val:.1f} → Score {score:+.2f}")

## 4. Composite Score Calculation

Calculate the weighted composite score from all metrics.

In [ ]:
def calculate_composite_score(data: dict, date: pd.Timestamp) -> dict:
    """
    Calculate composite score for a given date.
    Returns dict with individual scores and weighted composite.
    """
    scores = {}
    weighted_sum = 0
    total_weight = 0
    
    for metric, config in ALL_CONFIG.items():
        if metric not in data or data[metric].empty:
            continue
        
        df = data[metric]
        # Get value for date (or nearest prior)
        try:
            if date in df.index:
                value = df.loc[date, 'value']
            else:
                # Find nearest prior date
                prior = df.index[df.index <= date]
                if len(prior) > 0:
                    value = df.loc[prior[-1], 'value']
                else:
                    continue
        except:
            continue
        
        score = score_metric(value, config)
        weight = config['weight']
        
        scores[metric] = {
            'value': value,
            'score': score,
            'weight': weight,
            'weighted_score': score * weight
        }
        
        weighted_sum += score * weight
        total_weight += weight
    
    composite = weighted_sum / total_weight if total_weight > 0 else 0
    
    return {
        'date': date,
        'composite_score': composite,
        'individual_scores': scores,
        'total_weight': total_weight
    }

# Calculate for most recent date
latest_date = data['price'].index.max()
result = calculate_composite_score(data, latest_date)

print(f"\n{'='*60}")
print(f"COMPOSITE SCORE: {result['composite_score']:+.3f}")
print(f"Date: {result['date'].date()}")
print(f"{'='*60}\n")

print("Individual Metric Scores:")
print(f"{'Metric':<20} {'Value':>10} {'Score':>8} {'Weight':>8} {'Wtd Score':>10}")
print("-" * 60)
for metric, s in result['individual_scores'].items():
    print(f"{metric:<20} {s['value']:>10.4f} {s['score']:>+8.2f} {s['weight']:>8.0%} {s['weighted_score']:>+10.3f}")

## 5. Historical Composite Score

Calculate composite scores across all historical data for backtesting.

In [ ]:
def calculate_historical_composite(data: dict) -> pd.DataFrame:
    """Calculate composite score for all available dates."""
    # Get common date range
    price_df = data.get('price', pd.DataFrame())
    if price_df.empty:
        return pd.DataFrame()
    
    dates = price_df.index
    
    results = []
    for date in dates:
        result = calculate_composite_score(data, date)
        results.append({
            'date': date,
            'composite_score': result['composite_score'],
            'price': price_df.loc[date, 'value'] if date in price_df.index else np.nan
        })
    
    df = pd.DataFrame(results).set_index('date')
    return df

print("Calculating historical composite scores...")
historical = calculate_historical_composite(data)
print(f"Calculated {len(historical)} days of composite scores")
print(f"Date range: {historical.index.min().date()} to {historical.index.max().date()}")
historical.tail(10)

## 6. Visualize Composite Score vs Price

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price chart
ax1.semilogy(historical.index, historical['price'], color='orange', linewidth=1.5, label='BTC Price')
ax1.set_ylabel('Price (USD)', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')
ax1.set_title('Bitcoin Price & Checkmate Composite Score', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Composite score
colors = historical['composite_score'].apply(
    lambda x: '#22c55e' if x < -0.5 else '#ef4444' if x > 0.5 else '#fbbf24'
)
ax2.bar(historical.index, historical['composite_score'], color=colors, width=1, alpha=0.7)
ax2.axhline(y=0, color='white', linestyle='-', alpha=0.5)
ax2.axhline(y=-0.5, color='#22c55e', linestyle='--', alpha=0.5, label='Accumulate Zone')
ax2.axhline(y=0.5, color='#ef4444', linestyle='--', alpha=0.5, label='Distribute Zone')
ax2.axhline(y=-1.0, color='#22c55e', linestyle=':', alpha=0.3)
ax2.axhline(y=1.0, color='#ef4444', linestyle=':', alpha=0.3)
ax2.fill_between(historical.index, -2, -0.5, alpha=0.1, color='#22c55e')
ax2.fill_between(historical.index, 0.5, 2, alpha=0.1, color='#ef4444')
ax2.set_ylabel('Composite Score')
ax2.set_xlabel('Date')
ax2.set_ylim(-2, 2)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Add zone labels
ax2.text(historical.index[-1], -1.5, 'ACCUMULATE', fontsize=10, color='#22c55e', ha='right', va='center')
ax2.text(historical.index[-1], 1.5, 'DISTRIBUTE', fontsize=10, color='#ef4444', ha='right', va='center')

plt.tight_layout()
plt.show()

## 7. Trading Signal Generation

Implement the systematic trading rules from the framework.

In [ ]:
def generate_signals(data: dict, historical: pd.DataFrame) -> pd.DataFrame:
    """
    Generate trading signals based on the Checkmate Framework rules.
    
    Entry Rules (Accumulation Phase):
    - Primary: Composite < -0.5, MVRV < 1.2, SOPR < 1.02, LTH supply rising
    - Aggressive: MVRV < 0.8, STH-MVRV < 0.75, NUPL < 0
    
    Exit Rules (Distribution Phase):
    - Initial: Composite > 0.75, MVRV > 2.0, LTH-MVRV > 2.5
    - Aggressive: MVRV > 2.5, STH-MVRV > 1.3, NUPL > 0.6
    """
    signals = historical.copy()
    
    # Merge in required metrics
    for metric in ['mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'supply_lth']:
        if metric in data and not data[metric].empty:
            signals = signals.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
            signals[metric] = signals[metric].ffill()
    
    # Calculate LTH supply change (30d)
    if 'supply_lth' in signals.columns:
        signals['lth_supply_change_30d'] = signals['supply_lth'].pct_change(30)
    
    # === ENTRY SIGNALS ===
    
    # Primary entry conditions (all must be met)
    signals['entry_primary'] = (
        (signals['composite_score'] < -0.5) &
        (signals.get('mvrv', pd.Series(2.0)) < 1.2) &
        (signals.get('sopr', pd.Series(1.0)) < 1.02)
    )
    
    # Aggressive entry (any one triggers)
    signals['entry_aggressive'] = (
        (signals.get('mvrv', pd.Series(2.0)) < 0.8) |
        (signals.get('mvrv_sth', pd.Series(1.0)) < 0.75) |
        (signals.get('nupl', pd.Series(0.5)) < 0)
    )
    
    # === EXIT SIGNALS ===
    
    # Initial distribution (start taking profits)
    signals['exit_initial'] = (
        (signals['composite_score'] > 0.75) &
        (signals.get('mvrv', pd.Series(1.0)) > 2.0)
    )
    
    # Aggressive distribution (scale out significantly)
    signals['exit_aggressive'] = (
        (signals.get('mvrv', pd.Series(1.0)) > 2.5) |
        (signals.get('mvrv_sth', pd.Series(1.0)) > 1.3) |
        (signals.get('nupl', pd.Series(0.5)) > 0.6)
    )
    
    # === POSITION SIZING ===
    def get_position_size(row):
        score = row['composite_score']
        if score <= -1.5:
            return 1.0    # 100% - Deep Value
        elif score <= -1.0:
            return 0.75   # 75% - Value Zone
        elif score <= -0.5:
            return 0.50   # 50% - Mild Value
        elif score <= 0.5:
            return 0.25   # 25% - Neutral
        elif score <= 1.0:
            return 0.0    # 0% - Mild Risk (hold)
        elif score <= 1.5:
            return -0.25  # -25% (take profits)
        else:
            return -0.50  # -50% (aggressive distribution)
    
    signals['position_size'] = signals.apply(get_position_size, axis=1)
    
    # === SIGNAL SUMMARY ===
    def get_signal(row):
        if row.get('entry_aggressive', False):
            return 'STRONG BUY'
        elif row.get('entry_primary', False):
            return 'BUY'
        elif row.get('exit_aggressive', False):
            return 'STRONG SELL'
        elif row.get('exit_initial', False):
            return 'SELL'
        else:
            return 'HOLD'
    
    signals['signal'] = signals.apply(get_signal, axis=1)
    
    return signals

signals = generate_signals(data, historical)
print(f"\nSignal Distribution:")
print(signals['signal'].value_counts())
print(f"\nCurrent Signal: {signals['signal'].iloc[-1]}")
print(f"Position Size: {signals['position_size'].iloc[-1]:.0%}")

## 8. Signal Visualization

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price with buy/sell signals
ax1.semilogy(signals.index, signals['price'], color='white', linewidth=1, alpha=0.7)

# Mark buy signals
buy_mask = signals['signal'].isin(['BUY', 'STRONG BUY'])
ax1.scatter(signals.index[buy_mask], signals['price'][buy_mask], 
            color='#22c55e', marker='^', s=50, label='Buy', zorder=5)

# Mark sell signals
sell_mask = signals['signal'].isin(['SELL', 'STRONG SELL'])
ax1.scatter(signals.index[sell_mask], signals['price'][sell_mask], 
            color='#ef4444', marker='v', s=50, label='Sell', zorder=5)

ax1.set_ylabel('Price (USD)')
ax1.set_title('Checkmate Framework: Trading Signals', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Composite score
colors = signals['composite_score'].apply(
    lambda x: '#22c55e' if x < -0.5 else '#ef4444' if x > 0.5 else '#fbbf24'
)
ax2.bar(signals.index, signals['composite_score'], color=colors, width=1, alpha=0.7)
ax2.axhline(y=0, color='white', linestyle='-', alpha=0.5)
ax2.axhline(y=-0.5, color='#22c55e', linestyle='--', alpha=0.5)
ax2.axhline(y=0.75, color='#ef4444', linestyle='--', alpha=0.5)
ax2.set_ylabel('Composite Score')
ax2.set_ylim(-2, 2)
ax2.grid(True, alpha=0.3)

# Position sizing
ax3.fill_between(signals.index, 0, signals['position_size'], 
                 where=signals['position_size'] >= 0, 
                 color='#22c55e', alpha=0.5, label='Long Allocation')
ax3.fill_between(signals.index, 0, signals['position_size'], 
                 where=signals['position_size'] < 0, 
                 color='#ef4444', alpha=0.5, label='Take Profits')
ax3.axhline(y=0, color='white', linestyle='-', alpha=0.5)
ax3.set_ylabel('Position Size')
ax3.set_xlabel('Date')
ax3.set_ylim(-0.6, 1.1)
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Backtest Performance

Test the framework's historical performance.

In [ ]:
def backtest_framework(signals: pd.DataFrame, initial_capital: float = 100000) -> pd.DataFrame:
    """
    Backtest the Checkmate Framework.
    
    Strategy:
    - Allocate capital based on position_size signal
    - Rebalance daily (simplified)
    - Compare to buy-and-hold
    """
    bt = signals[['price', 'composite_score', 'position_size', 'signal']].copy()
    bt = bt.dropna(subset=['price'])
    
    # Calculate returns
    bt['returns'] = bt['price'].pct_change()
    
    # Strategy returns (position size * market returns)
    # Clip position between 0 and 1 for long-only
    bt['strategy_position'] = bt['position_size'].clip(0, 1).shift(1)  # Use previous day's signal
    bt['strategy_returns'] = bt['strategy_position'] * bt['returns']
    
    # Cumulative returns
    bt['hodl_equity'] = initial_capital * (1 + bt['returns']).cumprod()
    bt['strategy_equity'] = initial_capital * (1 + bt['strategy_returns']).cumprod()
    
    # Drawdowns
    bt['hodl_peak'] = bt['hodl_equity'].cummax()
    bt['hodl_drawdown'] = (bt['hodl_equity'] - bt['hodl_peak']) / bt['hodl_peak']
    
    bt['strategy_peak'] = bt['strategy_equity'].cummax()
    bt['strategy_drawdown'] = (bt['strategy_equity'] - bt['strategy_peak']) / bt['strategy_peak']
    
    return bt

backtest = backtest_framework(signals)

# Performance metrics
print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)

# Calculate stats
hodl_return = (backtest['hodl_equity'].iloc[-1] / 100000 - 1) * 100
strategy_return = (backtest['strategy_equity'].iloc[-1] / 100000 - 1) * 100

hodl_max_dd = backtest['hodl_drawdown'].min() * 100
strategy_max_dd = backtest['strategy_drawdown'].min() * 100

years = (backtest.index[-1] - backtest.index[0]).days / 365
hodl_cagr = ((backtest['hodl_equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100
strategy_cagr = ((backtest['strategy_equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100

print(f"\nPeriod: {backtest.index[0].date()} to {backtest.index[-1].date()} ({years:.1f} years)")
print(f"\n{'Metric':<25} {'HODL':>15} {'Strategy':>15}")
print("-" * 55)
print(f"{'Total Return':<25} {hodl_return:>14.1f}% {strategy_return:>14.1f}%")
print(f"{'CAGR':<25} {hodl_cagr:>14.1f}% {strategy_cagr:>14.1f}%")
print(f"{'Max Drawdown':<25} {hodl_max_dd:>14.1f}% {strategy_max_dd:>14.1f}%")
print(f"{'Final Equity':<25} ${backtest['hodl_equity'].iloc[-1]:>13,.0f} ${backtest['strategy_equity'].iloc[-1]:>13,.0f}")

In [ ]:
# Equity curve comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.semilogy(backtest.index, backtest['hodl_equity'], color='orange', linewidth=2, label='HODL', alpha=0.7)
ax1.semilogy(backtest.index, backtest['strategy_equity'], color='#22c55e', linewidth=2, label='Checkmate Strategy')
ax1.set_ylabel('Equity ($)')
ax1.set_title('Checkmate Framework Backtest: Equity Curves', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2.fill_between(backtest.index, backtest['hodl_drawdown']*100, 0, 
                 color='orange', alpha=0.3, label='HODL Drawdown')
ax2.fill_between(backtest.index, backtest['strategy_drawdown']*100, 0, 
                 color='#22c55e', alpha=0.5, label='Strategy Drawdown')
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Current Market Assessment

Detailed breakdown of the current market state according to the framework.

In [ ]:
def print_market_assessment(data: dict):
    """Print detailed current market assessment."""
    latest_date = data['price'].index.max()
    result = calculate_composite_score(data, latest_date)
    
    print("\n" + "="*70)
    print("CHECKMATE FRAMEWORK - CURRENT MARKET ASSESSMENT")
    print(f"Date: {latest_date.date()}")
    print("="*70)
    
    # Get price
    price = data['price'].loc[latest_date, 'value']
    print(f"\nBTC Price: ${price:,.0f}")
    
    # Composite score interpretation
    score = result['composite_score']
    if score <= -1.5:
        state, action = "DEEP VALUE", "Aggressive Accumulation (100%)"
    elif score <= -1.0:
        state, action = "VALUE ZONE", "Accumulation (75%)"
    elif score <= -0.5:
        state, action = "MILD VALUE", "DCA / Hold (50%)"
    elif score <= 0.5:
        state, action = "NEUTRAL", "Hold / Small DCA (25%)"
    elif score <= 1.0:
        state, action = "MILD RISK", "Reduce DCA, Hold"
    elif score <= 1.5:
        state, action = "RISK ZONE", "Distribution (Take 25-50%)"
    else:
        state, action = "EXTREME RISK", "Aggressive Distribution (Take 50-75%)"
    
    print(f"\nComposite Score: {score:+.3f}")
    print(f"Market State: {state}")
    print(f"Recommended Action: {action}")
    
    # Layer breakdown
    print("\n" + "-"*70)
    print("LAYER 1: INVESTOR PROFITABILITY")
    print("-"*70)
    for metric in LAYER1_CONFIG.keys():
        if metric in result['individual_scores']:
            s = result['individual_scores'][metric]
            signal = "🟢" if s['score'] < -0.5 else "🔴" if s['score'] > 0.5 else "🟡"
            print(f"{signal} {metric:<15} Value: {s['value']:>8.3f}  Score: {s['score']:>+6.2f}")
    
    print("\n" + "-"*70)
    print("LAYER 2: SPENDING BEHAVIOR")
    print("-"*70)
    for metric in LAYER2_CONFIG.keys():
        if metric in result['individual_scores']:
            s = result['individual_scores'][metric]
            signal = "🟢" if s['score'] < -0.5 else "🔴" if s['score'] > 0.5 else "🟡"
            print(f"{signal} {metric:<15} Value: {s['value']:>8.4f}  Score: {s['score']:>+6.2f}")
    
    print("\n" + "-"*70)
    print("LAYER 3: MARKET STRUCTURE")
    print("-"*70)
    for metric in LAYER3_CONFIG.keys():
        if metric in result['individual_scores']:
            s = result['individual_scores'][metric]
            signal = "🟢" if s['score'] < -0.5 else "🔴" if s['score'] > 0.5 else "🟡"
            print(f"{signal} {metric:<15} Value: {s['value']:>8.3f}  Score: {s['score']:>+6.2f}")
    
    print("\n" + "="*70)

print_market_assessment(data)

## 11. Key Pricing Anchors

Important price levels based on on-chain cost basis models.

In [ ]:
def print_price_anchors(data: dict):
    """Print key pricing anchors."""
    print("\n" + "="*60)
    print("KEY PRICING ANCHORS")
    print("="*60)
    
    # Current price
    price = data['price'].iloc[-1]['value']
    print(f"\nCurrent Price: ${price:,.0f}")
    
    anchors = [
        ('Realized Price', 'realized_price', 'Aggregate cost basis (MVRV floor)'),
        ('STH Realized Price', 'realized_price_sth', 'Short-term holder breakeven'),
        ('LTH Realized Price', 'realized_price_lth', 'Long-term holder breakeven'),
    ]
    
    print(f"\n{'Price Model':<25} {'Value':>12} {'vs Spot':>10} {'Significance'}")
    print("-" * 80)
    
    for name, metric, sig in anchors:
        if metric in data and not data[metric].empty:
            val = data[metric].iloc[-1]['value']
            pct = (price / val - 1) * 100
            print(f"{name:<25} ${val:>11,.0f} {pct:>+9.1f}%  {sig}")
    
    # MVRV multiples
    print("\n" + "-"*60)
    print("MVRV-Based Price Targets")
    print("-"*60)
    
    if 'realized_price' in data and not data['realized_price'].empty:
        rp = data['realized_price'].iloc[-1]['value']
        targets = [
            (1.0, 'Floor (MVRV = 1)'),
            (1.5, 'Fair Value'),
            (2.0, 'Initial Distribution'),
            (2.5, 'Aggressive Distribution'),
            (3.0, 'Euphoria Zone'),
        ]
        
        print(f"\n{'MVRV Multiple':>15} {'Target Price':>15} {'vs Current':>12}")
        for mult, label in targets:
            target = rp * mult
            vs_current = (target / price - 1) * 100
            print(f"{mult:.1f}x ({label}): ${target:>13,.0f} {vs_current:>+11.1f}%")

print_price_anchors(data)

## 12. Summary & Recommendations

Final summary and action items.

In [ ]:
# Final summary
latest = signals.iloc[-1]

print("\n" + "#"*70)
print("#" + " "*68 + "#")
print("#" + "   CHECKMATE FRAMEWORK SUMMARY".center(68) + "#")
print("#" + " "*68 + "#")
print("#"*70)

print(f"\n📅 Date: {latest.name.date()}")
print(f"💰 BTC Price: ${latest['price']:,.0f}")
print(f"📊 Composite Score: {latest['composite_score']:+.3f}")
print(f"📈 Signal: {latest['signal']}")
print(f"💼 Position Size: {latest['position_size']:.0%}")

# Key observations
print("\n" + "-"*70)
print("KEY OBSERVATIONS:")
print("-"*70)

observations = []
if 'mvrv' in latest and latest['mvrv'] < 1.2:
    observations.append("✅ MVRV below 1.2 - aggregate holders near/underwater")
elif 'mvrv' in latest and latest['mvrv'] > 2.0:
    observations.append("⚠️ MVRV above 2.0 - elevated unrealized profits")

if 'mvrv_sth' in latest and latest['mvrv_sth'] < 1.0:
    observations.append("✅ STH-MVRV below 1.0 - short-term holders underwater")
elif 'mvrv_sth' in latest and latest['mvrv_sth'] > 1.3:
    observations.append("⚠️ STH-MVRV above 1.3 - speculators in profit")

if 'sopr' in latest and latest['sopr'] < 1.0:
    observations.append("✅ SOPR below 1.0 - coins selling at a loss")
elif 'sopr' in latest and latest['sopr'] > 1.05:
    observations.append("⚠️ SOPR above 1.05 - elevated profit-taking")

if 'nupl' in latest and latest['nupl'] < 0.25:
    observations.append("✅ NUPL below 0.25 - market near capitulation")
elif 'nupl' in latest and latest['nupl'] > 0.6:
    observations.append("⚠️ NUPL above 0.6 - greed zone")

for obs in observations:
    print(f"  {obs}")

if not observations:
    print("  🟡 Market in neutral territory - no strong signals")

print("\n" + "-"*70)
print("FRAMEWORK RECOMMENDATION:")
print("-"*70)

if latest['composite_score'] < -0.5:
    print("  🟢 VALUE TERRITORY - Consider accumulation on weakness")
elif latest['composite_score'] > 0.75:
    print("  🔴 RISK TERRITORY - Consider taking profits")
else:
    print("  🟡 NEUTRAL TERRITORY - Hold current positions, light DCA acceptable")

print("\n" + "#"*70)

---

*"We cannot predict the future, but we can certainly prepare for it."* - James Check

---